# Phase 1D.3 Colab run

Use a fresh Tesla T4 runtime. This notebook stops on environment or wheel provenance mismatches. It does not create a public tunnel. Save cell outputs as evidence.

In [ ]:
import hashlib, json, os, subprocess
from pathlib import Path
REPO_URL='https://github.com/MoeMo-s-Team/DACN_HeThongHoTroTuhoc_PhanHeNguoiDung.git'
REVIEWED_COMMIT=os.environ.get('PHASE1D3_REVIEWED_COMMIT')
if not REVIEWED_COMMIT or len(REVIEWED_COMMIT)!=40: raise RuntimeError('Set PHASE1D3_REVIEWED_COMMIT to the reviewed 40-character commit.')
REPO_BASE=Path('/content/phase-1d3-repo-'+REVIEWED_COMMIT[:8])
REPO=REPO_BASE
suffix=0
while REPO.exists():
    suffix+=1
    REPO=Path(f'{REPO_BASE}-{suffix}')
clone=subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO)],capture_output=True,text=True)
if clone.returncode: raise RuntimeError(f'git clone failed ({clone.returncode}):\n{clone.stderr}')
subprocess.run(['git','checkout','--detach',REVIEWED_COMMIT],cwd=REPO,check=True)
actual=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
dirty=subprocess.run(['git','status','--porcelain','--untracked-files=all'],cwd=REPO,capture_output=True,text=True,check=True).stdout
if actual!=REVIEWED_COMMIT or dirty: raise RuntimeError('Reviewed source checkout mismatch or dirty tree.')
proof={'verified':True,'commit':actual,'repo':str(REPO),'tree':subprocess.run(['git','rev-parse','HEAD^{tree}'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()}
proof_path=Path('/content/dacn-source-proof.json'); proof_path.write_text(json.dumps(proof,indent=2)+'\n')
print(proof)


In [ ]:
import sys
sys.path.insert(0,str(REPO))
os.chdir(REPO)
from ai.colab.torch_provenance import verify_source_proof
proof_commit=verify_source_proof('/content/dacn-source-proof.json',REPO)
if proof_commit!=REVIEWED_COMMIT: raise RuntimeError('BLOCKED: source proof differs from requested reviewed commit.')
print('Verified clean source commit:',proof_commit)


In [ ]:
# Snapshot before any dependency operation. Never install/replace Torch.
import json, platform, torch, hashlib
snapshot = {'python':platform.python_version(),'pythonExecutable':sys.executable,'torchRoot':str(Path(torch.__file__).parent.parent),'torch':torch.__version__,'cuda':torch.version.cuda,
 'pipFreeze':subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout.splitlines(),
 'torchFiles':{}, 'gpu':subprocess.run(['nvidia-smi','--query-gpu=name,uuid,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()}

for rel in __import__('importlib.metadata').metadata.distribution('torch').files:
    path=Path(torch.__file__).parent.parent / rel
    if path.is_file(): snapshot['torchFiles'][str(rel)]=hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(snapshot,indent=2))
if not snapshot['python'].startswith('3.13.') or snapshot['torch']!='2.11.0+cu130' or snapshot['cuda']!='13.0' or 'Tesla T4' not in snapshot['gpu']:
    raise RuntimeError('BLOCKED: Python/Torch/CUDA/T4 candidate mismatch; stop without changing Torch.')
Path('/content/dacn-colab-before.json').write_text(json.dumps(snapshot,indent=2)+'\n')

In [ ]:

from pathlib import Path
import json

snapshot_path = Path("/content/dacn-colab-before.json")
repo = Path("/content/phase-1d3-repo")

assert snapshot_path.is_file(), "Snapshot chưa được lưu"

data = json.loads(snapshot_path.read_text())

print("SNAPSHOT: PASS")
print("Python:", data["python"])
print("Torch:", data["torch"])
print("CUDA:", data["cuda"])
print("GPU:", data["gpu"].split(",")[0])
print("Torch files hashed:", len(data["torchFiles"]))

print("\nDEPENDENCY GATE:")
print("requirements.lock:", (repo / "ai/requirements.lock").exists())
print("Torch wheel provenance:", Path("/content/torch-wheel-provenance.json").exists())


In [ ]:
provenance_run=subprocess.run([sys.executable,str(REPO/'ai/colab/torch_provenance.py'),'--repo',str(REPO),'--snapshot','/content/dacn-colab-before.json','--source-proof','/content/dacn-source-proof.json','--output','/content/torch-wheel-provenance.json'],capture_output=True,text=True)
if provenance_run.stdout: print(provenance_run.stdout)
if provenance_run.returncode: raise RuntimeError(f'Torch provenance blocked (exit {provenance_run.returncode}):\n{provenance_run.stderr}')


## Dependency lock gate

After the provenance cell passes, create a **separate resolver** and lock outside the source checkout:

```bash
python3.13 -m venv /content/phase1d3-lock-resolver
/content/phase1d3-lock-resolver/bin/python -m pip install pip-tools==7.5.2
PIP_EXTRA_INDEX_URL=https://download.pytorch.org/whl/cu130 /content/phase1d3-lock-resolver/bin/pip-compile --generate-hashes --output-file /content/ai-requirements.lock --constraint /content/phase-1d3-repo/ai/constraints-colab.txt /content/phase-1d3-repo/ai/requirements.in
```

Do not install this lock here. The next gate checks its Torch hash against the wheel hash verified from the installed Torch RECORD, validates source proof, and runs pip dry-run; any missing or mismatched evidence stops before package installation. Keep resolver output and evidence under `/content`, separate from the clean reviewed source checkout.


In [ ]:
from ai.colab.preflight import validate_install_plan, verify_torch_wheel, lock_contains_torch_hash
from ai.colab.torch_provenance import verify_source_proof
def installed_torch_files():
    distribution=__import__('importlib.metadata').metadata.distribution('torch')
    root=Path(__import__('torch').__file__).parent.parent
    return {str(rel):hashlib.sha256((root/rel).read_bytes()).hexdigest() for rel in distribution.files if (root/rel).is_file()}
subprocess.run([sys.executable,'-m','venv','--system-site-packages','/content/dacn-venv'],check=True)
VENV_PY='/content/dacn-venv/bin/python'
provenance_path=Path('/content/torch-wheel-provenance.json'); source_proof_path=Path('/content/dacn-source-proof.json')
lock=Path('/content/ai-requirements.lock'); plan=Path('/content/dacn-install-plan.json')
if not source_proof_path.is_file() or not provenance_path.is_file() or not lock.is_file(): raise RuntimeError('BLOCKED: source proof, Torch provenance and resolver lock are required.')
source_proof=json.loads(source_proof_path.read_text()); provenance=json.loads(provenance_path.read_text())
actual_commit=verify_source_proof(source_proof_path,REPO)
if not source_proof.get('verified') or source_proof.get('commit')!=actual_commit or provenance.get('sourceCommit')!=actual_commit: raise RuntimeError('BLOCKED: source proof/provenance commit mismatch.')
if provenance.get('sourceProofSha256')!=hashlib.sha256(source_proof_path.read_bytes()).hexdigest(): raise RuntimeError('BLOCKED: provenance source-proof hash mismatch.')
if provenance.get('version')!=snapshot['torch'] or provenance.get('sha256') is None or not verify_torch_wheel(provenance.get('wheelPath',''),provenance.get('officialUrl',''),provenance.get('sha256',''),snapshot['torchFiles'],snapshot['pythonExecutable'],snapshot['torchRoot']): raise RuntimeError('BLOCKED: Torch wheel hash or installed RECORD mismatch.')
lock_text=lock.read_text()
if not lock_contains_torch_hash(lock_text,provenance['sha256']): raise RuntimeError('BLOCKED: lock is malformed or omits the verified Torch wheel hash.')
if installed_torch_files()!=snapshot['torchFiles']: raise RuntimeError('BLOCKED: installed Torch changed since the snapshot.')
subprocess.run([VENV_PY,'-m','pip','install','--dry-run','--report',str(plan),'-r',str(lock),'-c',str(REPO/'ai/constraints-colab.txt')],check=True)
if not validate_install_plan(json.loads(plan.read_text())): raise RuntimeError('BLOCKED: dry-run would install/remove/replace Torch.')
if verify_source_proof(source_proof_path,REPO)!=actual_commit or installed_torch_files()!=snapshot['torchFiles']: raise RuntimeError('BLOCKED: source or Torch changed during dry-run.')
print('PASS: source, wheel RECORD, lock hash and no-Torch-action dry-run. No packages installed.')


In [ ]:
# Partial preflight: modelLoad stays false until the real worker starts under the GPU lock.
from ai.colab.preflight import preflight_environment
pip_check=subprocess.run([VENV_PY,'-m','pip','check'],capture_output=True,text=True)
if pip_check.returncode: raise RuntimeError('BLOCKED: pip check failed: '+pip_check.stdout+pip_check.stderr)
probe="import json,torch,transformers,bitsandbytes; cfg=transformers.BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16); print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'nf4Import':isinstance(cfg,transformers.BitsAndBytesConfig)}))"
probe_result=json.loads(subprocess.run([VENV_PY,'-c',probe],capture_output=True,text=True,check=True).stdout)
checks={'python':snapshot['python'],'torch':probe_result['torch'],'cuda':probe_result['cuda'],'gpu':'Tesla T4',
        'pipCheck':pip_check.returncode==0,'nf4Import':probe_result['nf4Import'],'modelLoad':False}
print(json.dumps(preflight_environment(checks),indent=2))

## Run benchmark

Keep Uvicorn local. Supply the bearer token through a hidden prompt. Do not create a tunnel here; establish authorized private reachability separately.

In [ ]:
from getpass import getpass
import hashlib, time
from ai.colab.preflight import preflight_environment
os.environ['AI_RUNTIME_TOKEN']=getpass('Runtime bearer token (hidden): ')
os.environ['AI_RUNTIME_MODEL_ID']='Qwen/Qwen3-4B-Instruct-2507'
os.environ['AI_RUNTIME_MODEL_REVISION']='cdbee75f17c01a7cc42f958dc650907174af0554'
os.environ['AI_RUNTIME_GPU_UUID']=snapshot['gpu'].split(',')[1].strip()
os.environ['AI_BENCHMARK_GPU']=snapshot['gpu'].split(',')[0].strip()
os.environ['AI_BENCHMARK_GPU_UUID']=snapshot['gpu'].split(',')[1].strip()
os.environ['AI_BENCHMARK_TORCH']=snapshot['torch']
os.environ['AI_BENCHMARK_CUDA']=snapshot['cuda']
from ai.benchmark.allowlist import fixture_request_hash
fixture_rows=[json.loads(line) for line in (REPO/'ai/benchmark/fixtures/fixtures.jsonl').read_text().splitlines()]
os.environ['AI_RUNTIME_BENCHMARK_MODE']='1'
os.environ['AI_RUNTIME_BENCHMARK_ALLOWLIST']=','.join(sorted({fixture_request_hash(row['task'],row['input']) for row in fixture_rows}))
os.environ['AI_RUNTIME_OBSERVATION_DIR']='/content/dacn-raw-observations'
node_version=subprocess.run(['node','--version'],capture_output=True,text=True,check=True).stdout.strip().lstrip('v')
if not node_version.startswith('24.') or tuple(map(int,node_version.split('.')[:2]))<(24,18): raise RuntimeError('Node must be >=24.18.0 and <25.')
pnpm_version=subprocess.run(['pnpm','--version'],capture_output=True,text=True,check=True).stdout.strip()
if pnpm_version!='10.12.1': raise RuntimeError('pnpm must be 10.12.1.')
server=subprocess.Popen([VENV_PY,'-m','uvicorn','ai.runtime.app:app','--host','127.0.0.1','--port','8000'],cwd=REPO,stdout=open('/content/dacn-runtime.log','w'),stderr=subprocess.STDOUT)
for _ in range(600):
    if server.poll() is not None: raise RuntimeError('Runtime/model startup failed; inspect local log.')
    log=Path('/content/dacn-runtime.log').read_text() if Path('/content/dacn-runtime.log').exists() else ''
    if 'Application startup complete' in log: break
    time.sleep(1)
else: server.terminate(); raise RuntimeError('Runtime/model startup timed out.')
checks['modelLoad']=True
result=preflight_environment(checks); print(json.dumps(result,indent=2))
if result['status']!='READY': server.terminate(); raise RuntimeError('BLOCKED: model/runtime preflight failed.')
subprocess.run([VENV_PY,'ai/colab/measure_capacity.py'],cwd=REPO,check=True)
subprocess.run([VENV_PY,'ai/benchmark/validate_fixtures.py'],cwd=REPO,check=True)
os.environ['AI_BENCHMARK_MODE']='1'
from ai.runtime.prompts import PROMPT_VERSIONS
os.environ['AI_BENCHMARK_CONFIG_HASH']=hashlib.sha256(json.dumps({'model':'Qwen/Qwen3-4B-Instruct-2507','revision':'cdbee75f17c01a7cc42f958dc650907174af0554','torch':snapshot['torch'],'cuda':snapshot['cuda'],'promptVersions':PROMPT_VERSIONS},sort_keys=True).encode()).hexdigest()
run_result=subprocess.run([VENV_PY,'ai/benchmark/run.py','--fixtures','ai/benchmark/fixtures/fixtures.jsonl','--endpoint','http://127.0.0.1:8000/v1/infer','--output-dir','/content/dacn-benchmark','--export-output','/content/dacn-benchmark/outputs'],cwd=REPO,capture_output=True,text=True,check=True)
run_dir=Path(json.loads(run_result.stdout)['runDir'])
print(f'Benchmark run directory: {run_dir}')

In [ ]:
# After manually adding scores.json, run this cell. It remains Pending until verified evidence.json is also present.
if 'run_dir' not in globals() or not (run_dir/'manifest.json').exists(): raise RuntimeError('Run the benchmark cell first; set run_dir to the intended resumed run when resuming.')
subprocess.run([VENV_PY,'ai/benchmark/report.py','--run-dir',str(run_dir)],cwd=REPO,check=True)
# Add one five-dimension score object per fixture to run_dir/scores.json before reporting final PASS eligibility.
import hashlib
artifact_roots=[Path('/content/dacn-benchmark'),Path('/content/dacn-source-proof.json'),Path(REPO/'ai/requirements.lock'),Path('/content/dacn-raw-observations'),Path('/content/dacn-capacity-measurements.json'),Path('/content/dacn-colab-before.json'),Path('/content/dacn-install-plan.json'),Path('/content/torch-wheel-provenance.json'),Path('/content/dacn-runtime.log')]
manifest={}
for item in artifact_roots:
    files=[item] if item.is_file() else ([f for f in item.rglob('*') if f.is_file()] if item.exists() else [])
    for file in files:
        if file.name!='SHA256SUMS.json': manifest[str(file)]=hashlib.sha256(file.read_bytes()).hexdigest()
Path('/content/dacn-phase1d3-SHA256SUMS.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('Evidence hash manifest: /content/dacn-phase1d3-SHA256SUMS.json')
import zipfile
archive='/content/dacn-phase1d3-evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
    for item in artifact_roots+[Path('/content/dacn-nestjs-transport-evidence.json')]:
        files=[item] if item.is_file() else ([f for f in item.rglob('*') if f.is_file()] if item.exists() else [])
        for file in files:
            if file.name!='SHA256SUMS.json': bundle.write(file,arcname=str(file).lstrip('/'))
    bundle.write('/content/dacn-phase1d3-SHA256SUMS.json',arcname='dacn-phase1d3-SHA256SUMS.json')
print('Return this archive:',archive)


## Return evidence

Review every output with ai/benchmark/rubric.md and save all five scores per attempted fixture. Return the full run directory, source hash, lock and wheel evidence, environment snapshot, fixture hash, attempts, evaluator results, scores, capacity measurements and NestJS transport evidence. Do not activate a profile or close OPEN-ARCH-003.